# 2.4 Wedding Planner (adaptado a lab local)

**Qué se ve aquí:** proyecto multi-agente: un coordinador con **state propio** (`WeddingState`) delega en 3 subagentes: vuelos (MCP remoto Kiwi), venues (Tavily) y playlist (SQL sobre Chinook.db).

> ⚠️ En CPU este notebook es **largo** (decenas de llamadas al modelo, `recursion_limit=40`). Cuenta con 15–40 min. Sigue el progreso en LangSmith (tag `WP`).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest", num_ctx=32768)   # Kiwi + Tavily devuelven mucho texto → 32K (ver M2.1b)

# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 1) added error handling for MCP failures, 2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


In [2]:
# Sin cambios
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [3]:
# 🟢 ADAPTADO LOCAL (celda nueva)
# Motivo: Kiwi expone también feedback-to-devs (envía texto a sus devs) → solo search-flight (ver Catálogo MCP).
tools = [t for t in tools if t.name == "search-flight"]
print([t.name for t in tools])

['search-flight']


In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: Tavily devuelve 5 resultados completos (~2.000 tokens) por búsqueda; en CPU con varias búsquedas el contexto explota.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from typing import Dict, Any
# from tavily import TavilyClient
# from langchain.tools import tool
#
# tavily_client = TavilyClient()
#
# @tool
# def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
#     """Search the web for information. You must track your search count by providing
#     search_number (starting at 1) and max_search_number on every call.
#     Queries must use only plain text characters. Do not use accented or special characters     
#       (e.g., use 'capacite' instead of 'capacité').
#     """
#     if search_number > max_search_number:
#         return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
#     try:
#         return tavily_client.search(query)
#     except Exception as e:
#         return {"error": str(e)}

In [4]:
# 🟢 ADAPTADO LOCAL
# Motivo: max_results=3 para bajar tokens por búsqueda (ver M1.4 / M1.7).
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query, max_results=3)
    except Exception as e:
        return {"error": str(e)}

In [5]:
# Sin cambios
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

/tmp/ipykernel_284911/4003213382.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Create State

In [6]:
# Sin cambios
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# # Travel agent
# travel_agent = create_agent(
#     model="gpt-5-nano",
#     tools=tools,
#     system_prompt="""
#     You are a travel agent. Search for flights to the desired destination wedding location.
#     You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
#     - Price (lowest, economy class)
#     - Duration (shortest)
#     - Date (time of year which you believe is best for a wedding at this location)
#     To make things easy, only look for one ticket, one way.
#     You may need to make multiple searches to iteratively find the best options.
#     You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
#     If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
#     Once you have found the best options, let the user know your shortlist of options.
#     """
# )

In [7]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model=MODEL,
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# # Venue agent
# venue_agent = create_agent(
#     model="gpt-5-nano",
#     tools=[web_search],
#     system_prompt="""
#     You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
#     You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
#     - Price (lowest)
#     - Capacity (exact match)
#     - Reviews (highest)
#     You may need to make multiple searches to iteratively find the best options. 
#     You have a suggested limit of 12 web searches. Count every web_search call you make.
#     After 12 searches, you should stop searching and summarize the best options you have
#     found so far.
#     """
# )

In [8]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
# Venue agent
venue_agent = create_agent(
    model=MODEL,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# # Playlist agent
# playlist_agent = create_agent(
#     model="gpt-5-nano",
#     tools=[query_playlist_db],
#     system_prompt="""
#     You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
#     Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
#     If you run into errors when querying the database, try to fix them by making changes to the query.
#     Do not come back empty handed, keep trying to query the db until you find a list of songs.
#
#     This is a SQLite database. Before writing any data queries, first discover the schema.
#     """
# )

In [9]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
# Playlist agent
playlist_agent = create_agent(
    model=MODEL,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [10]:
# Sin cambios
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# coordinator = create_agent(
#     model="gpt-5-nano",
#     tools=[search_flights, search_venues, suggest_playlist, update_state],
#     state_schema=WeddingState,
#     system_prompt="""
#     You are a wedding coordinator. 
#     First find all the information you need to update the state. When you have the information, update the state.
#     Once that has completed and returned, you can delegate the tasks 
#     to your specialists for flights, venues, and playlists.
#     Once you have received their answers, coordinate the perfect wedding for me.
#     """
# )

In [11]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

coordinator = create_agent(
    model=MODEL,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


In [12]:
# Sin cambios
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [13]:
# Sin cambios
from pprint import pprint

pprint(response)

{'destination': 'Paris',
 'genre': 'jazz',
 'guest_count': '100',
 'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='00620fb4-d1f3-40bd-870f-854381303fe3'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:53:37.8343357Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15086530300, 'load_duration': 6999221800, 'prompt_eval_count': 344, 'prompt_eval_duration': 6219317000, 'eval_count': 37, 'eval_duration': 1863186000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a1098d-12b0-7a23-9872-70625448152d-0', tool_calls=[{'name': 'update_state', 'args': {'destination': 'Paris', 'genre': 'jazz', 'guest_count': '100', 'origin': 'London'}, 'id': 'e91cfd6b-02cc-497d-bff6-76dbc6dd296f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

In [14]:
# Sin cambios
print(response["messages"][-1].content)

As your wedding coordinator, I have gathered all the necessary information and consulted with my specialists for flights, venues, and playlists.

Here is the complete coordination for your Jazz-themed wedding in Paris for 100 guests, traveling from London:

### ✈️ Flight Coordination (London to Paris)
**Status:** **Awaiting Date**
*   **Action Required:** I need a specific departure date in **dd/mm/yyyy** to search for the best flight options for you.

### 🏰 Venue Coordination (Paris)
**Status:** **Options Provided**
*   **Top Recommendation:** **Hôtel Raphaël Paris** is the most direct match for your **100-guest capacity** in a high-end Parisian setting.
*   **Alternative for Prestige:** **Hôtel Plaza Athénée** offers exceptional reviews, though it may be slightly smaller (up to 80 guests).
*   **Next Step:** I recommend contacting the venue of your choice to confirm availability for your desired date and to get a precise quote.

### 🎶 Playlist Coordination (Jazz)
**Status:** **Curate

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r